# Pass2022 Crossmatch to Gaia DR3
Resolves star names via SIMBAD, propagates proper motions to Gaia epoch (J2016.0),
then crossmatches to Gaia DR3 via cone search.

**Install deps if needed:**
```
pip install astroquery astropy numpy pandas tqdm
```

In [ ]:
import numpy as np
import pandas as pd
from astroquery.simbad import Simbad
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

XMATCH_RADIUS_ARCSEC = 5.0
RUWE_THRESHOLD = 1.4
GAIA_EPOCH = 2016.0

## Step 1: Load Pass2022 and build flat list of all stars

In [ ]:
df = pd.read_csv('cf_data/pass.csv')
print(f'Loaded {len(df)} systems')
display(df.head())

# Build flat list: one row per star (both components of each pair)
stars = []
for _, row in df.iterrows():
    for role, name_col, mass_col, prot_col in [
        ('primary',   'star_name',      'mass_msun',           'prot_days'),
        ('companion', 'companion_name', 'companion_mass_msun', 'companion_prot_days'),
    ]:
        stars.append({
            'name':                      row[name_col],
            'role':                      role,
            'binary_type':               row['binary_type'],
            'system_id':                 row['star_name'],
            'mass_msun':                 row[mass_col],
            'prot_days':                 row[prot_col],
            'age_min_gyr':               row.get('age_min_gyr', np.nan),
            'age_c18_gyr':               row.get('age_c18_gyr', np.nan),
            'flag_higher_order_multiple': row['flag_higher_order_multiple'],
        })

stars_df = pd.DataFrame(stars)
print(f'Total stars to crossmatch: {len(stars_df)}')
display(stars_df.head())

## Step 2: Resolve coordinates via SIMBAD

In [ ]:
custom_simbad = Simbad()
custom_simbad.add_votable_fields('pmra', 'pmdec', 'plx')

def resolve_simbad(name):
    try:
        result = custom_simbad.query_object(name)
        if result is None or len(result) == 0:
            return None
        row = result[0]
        return {
            'simbad_ra':       float(row['RA_d']),
            'simbad_dec':      float(row['DEC_d']),
            'simbad_pmra':     float(row['PMRA'])      if row['PMRA']      is not None else np.nan,
            'simbad_pmdec':    float(row['PMDEC'])     if row['PMDEC']     is not None else np.nan,
            'simbad_parallax': float(row['PLX_VALUE']) if row['PLX_VALUE'] is not None else np.nan,
        }
    except Exception as e:
        return None

print('Function defined.')

In [ ]:
simbad_results = []
for _, row in tqdm(stars_df.iterrows(), total=len(stars_df), desc='SIMBAD'):
    result = resolve_simbad(row['name'])
    if result is None:
        simbad_results.append({
            'simbad_ra': np.nan, 'simbad_dec': np.nan,
            'simbad_pmra': np.nan, 'simbad_pmdec': np.nan,
            'simbad_parallax': np.nan, 'simbad_resolved': False
        })
    else:
        result['simbad_resolved'] = True
        simbad_results.append(result)

stars_df = pd.concat([stars_df.reset_index(drop=True), pd.DataFrame(simbad_results)], axis=1)

n_resolved = stars_df['simbad_resolved'].sum()
print(f'Resolved: {n_resolved}/{len(stars_df)}')
print('Failed:')
print(stars_df[~stars_df['simbad_resolved']]['name'].tolist())

In [ ]:
# Retry failed names with alternate formats
# TIC IDs, 2MASS names sometimes need slight reformatting
failed_mask = ~stars_df['simbad_resolved']
print(f'Retrying {failed_mask.sum()} failed names...')

for idx, row in stars_df[failed_mask].iterrows():
    name = row['name']
    alternates = [
        f'TIC {name}',
        name.replace('2MASS ', '2MASS J'),
        name.replace('2MASSJ', '2MASS J'),
        name.replace(' ', ''),
    ]
    for alt in alternates:
        result = resolve_simbad(alt)
        if result is not None:
            print(f'  Resolved "{name}" via "{alt}"')
            for col, val in result.items():
                stars_df.at[idx, col] = val
            stars_df.at[idx, 'simbad_resolved'] = True
            break

print(f'\nFinal resolved: {stars_df["simbad_resolved"].sum()}/{len(stars_df)}')
print('Still unresolved:')
print(stars_df[~stars_df['simbad_resolved']]['name'].tolist())

## Step 3: Propagate proper motions to Gaia epoch (J2016.0)

In [ ]:
def propagate_coords(ra, dec, pmra, pmdec, parallax, from_epoch=2000.0, to_epoch=2016.0):
    if np.isnan(pmra) or np.isnan(pmdec):
        return ra, dec  # can't propagate without PM, use original coords
    dist = (1000.0 / parallax) * u.pc if (pd.notna(parallax) and parallax > 0) else None
    coord = SkyCoord(
        ra=ra * u.deg, dec=dec * u.deg,
        pm_ra_cosdec=pmra * u.mas/u.yr,
        pm_dec=pmdec * u.mas/u.yr,
        distance=dist,
        obstime=Time(from_epoch, format='decimalyear')
    )
    prop = coord.apply_space_motion(new_obstime=Time(to_epoch, format='decimalyear'))
    return float(prop.ra.deg), float(prop.dec.deg)

ra_prop, dec_prop = [], []
for _, row in stars_df.iterrows():
    if row['simbad_resolved']:
        ra, dec = propagate_coords(
            row['simbad_ra'], row['simbad_dec'],
            row['simbad_pmra'], row['simbad_pmdec'],
            row['simbad_parallax']
        )
    else:
        ra, dec = np.nan, np.nan
    ra_prop.append(ra)
    dec_prop.append(dec)

stars_df['ra_j2016'] = ra_prop
stars_df['dec_j2016'] = dec_prop
print('Done. Sample of propagated coords:')
print(stars_df[['name', 'simbad_ra', 'ra_j2016', 'simbad_dec', 'dec_j2016']].head(6).to_string())

## Step 4: Crossmatch to Gaia DR3

In [ ]:
def gaia_cone_search(ra_deg, dec_deg, radius_arcsec=5.0):
    r_deg = radius_arcsec / 3600.0
    query = f"""
    SELECT source_id, ra, dec,
           parallax, parallax_error, parallax_over_error,
           pmra, pmdec,
           phot_g_mean_mag, bp_rp,
           ebpminrp_gspphot, teff_gspphot, ruwe,
           DISTANCE(POINT({ra_deg},{dec_deg}), POINT(ra,dec)) AS ang_sep_deg
    FROM gaiadr3.gaia_source
    WHERE CONTAINS(POINT(ra,dec), CIRCLE({ra_deg},{dec_deg},{r_deg})) = 1
    ORDER BY ang_sep_deg ASC
    """
    try:
        result = Gaia.launch_job(query).get_results().to_pandas()
        result['ang_sep_arcsec'] = result['ang_sep_deg'] * 3600.0
        return result
    except:
        return pd.DataFrame()

EMPTY = {
    'n_gaia_matches': 0, 'gaia_source_id': pd.NA,
    'gaia_ra': pd.NA, 'gaia_dec': pd.NA, 'gaia_ang_sep_arcsec': pd.NA,
    'gaia_parallax': pd.NA, 'gaia_parallax_error': pd.NA, 'gaia_parallax_over_error': pd.NA,
    'gaia_pmra': pd.NA, 'gaia_pmdec': pd.NA,
    'gaia_phot_g_mean_mag': pd.NA, 'gaia_bp_rp': pd.NA,
    'gaia_ebpminrp_gspphot': pd.NA, 'gaia_teff_gspphot': pd.NA, 'gaia_ruwe': pd.NA,
    'flag_no_match': True, 'flag_multi_match': False,
    'flag_high_ruwe': False, 'flag_no_ebpminrp': True, 'bprp0': pd.NA,
}

gaia_records = []
for _, row in tqdm(stars_df.iterrows(), total=len(stars_df), desc='Gaia xmatch'):
    if not row['simbad_resolved'] or np.isnan(row['ra_j2016']):
        gaia_records.append(EMPTY.copy())
        continue

    matches = gaia_cone_search(row['ra_j2016'], row['dec_j2016'], XMATCH_RADIUS_ARCSEC)
    n = len(matches)

    if n == 0:
        gaia_records.append(EMPTY.copy())
    else:
        best = matches.iloc[0]
        ebp  = best['ebpminrp_gspphot']
        bprp = best['bp_rp']
        ruwe = best['ruwe']
        gaia_records.append({
            'n_gaia_matches':            n,
            'gaia_source_id':            int(best['source_id']),
            'gaia_ra':                   best['ra'],
            'gaia_dec':                  best['dec'],
            'gaia_ang_sep_arcsec':       best['ang_sep_arcsec'],
            'gaia_parallax':             best['parallax'],
            'gaia_parallax_error':       best['parallax_error'],
            'gaia_parallax_over_error':  best['parallax_over_error'],
            'gaia_pmra':                 best['pmra'],
            'gaia_pmdec':                best['pmdec'],
            'gaia_phot_g_mean_mag':      best['phot_g_mean_mag'],
            'gaia_bp_rp':                bprp,
            'gaia_ebpminrp_gspphot':     ebp,
            'gaia_teff_gspphot':         best['teff_gspphot'],
            'gaia_ruwe':                 ruwe,
            'flag_no_match':             False,
            'flag_multi_match':          n > 1,
            'flag_high_ruwe':            bool(ruwe >= RUWE_THRESHOLD) if pd.notna(ruwe) else False,
            'flag_no_ebpminrp':          pd.isna(ebp),
            'bprp0':                     (bprp - ebp) if (pd.notna(bprp) and pd.notna(ebp)) else pd.NA,
        })

result_df = pd.concat([stars_df.reset_index(drop=True), pd.DataFrame(gaia_records)], axis=1)
print(f'Done! {len(result_df)} rows')

## Step 5: Summary

In [ ]:
total = len(result_df)
print(f'Total stars:              {total}')
print(f'SIMBAD unresolved:        {(~result_df["simbad_resolved"]).sum()}')
print(f'No Gaia match:            {result_df["flag_no_match"].sum()}')
print(f'Multiple Gaia matches:    {result_df["flag_multi_match"].sum()}')
print(f'High RUWE (>={RUWE_THRESHOLD}):       {result_df["flag_high_ruwe"].sum()}')
print(f'Missing ebpminrp:         {result_df["flag_no_ebpminrp"].sum()}  <- cannot use in ChronoFlow')

print('\n--- Multi-match cases ---')
display(result_df[result_df['flag_multi_match']][['name','binary_type','n_gaia_matches','gaia_ang_sep_arcsec','gaia_ruwe','bprp0']])

print('\n--- No-match cases ---')
display(result_df[result_df['flag_no_match']][['name','binary_type','simbad_resolved','ra_j2016','dec_j2016']])

## Step 6: Save

In [ ]:
out_path = 'cf_data/pass2022_gaia_xmatch.csv'
result_df.to_csv(out_path, index=False)
print(f'Saved: {out_path} ({len(result_df)} rows, {len(result_df.columns)} cols)')